# Preprocessing and integration - STARWARS_AUTOCALLS

Builds a one-row-per-`rfq_id` feature table: explodes baskets by ticker, joins reference and historical volatility without future leakage, aggregates by RFQ, and creates categorical dummies.

`start_date` and `end_date` are not model features because they are not reliably available when pricing a new RFQ.

## Integration, dummy variables, and outputs


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# The notebook can run from notebooks/ or from the project root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data" / "raw").exists() else cwd.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1) Load and validate the three sources.
rfqs = pd.read_csv(RAW_DIR / "rfqs.csv", parse_dates=["requested_date", "start_date", "end_date"])
vol = pd.read_csv(RAW_DIR / "daily_volatility.csv", parse_dates=["date"])
reference = pd.read_csv(RAW_DIR / "underlyings_reference.csv")

assert rfqs["rfq_id"].is_unique, "rfq_id must be unique"
assert reference["underlying"].is_unique, "Reference tickers must be unique"
assert not vol.duplicated(["date", "underlying"]).any(), "Duplicate (date, ticker) market records"

# Trend of the volatility indicator at each market date. It compares the latest
# 63-day realized volatility with the 21 prior published observations for the same ticker.
# A positive value means volatility is above its recent level; negative means below it.
vol = vol.sort_values(["underlying", "date"]).copy()
vol["realized_vol_21d_prior_mean"] = (
    vol.groupby("underlying")["realized_vol_63d"]
    .transform(lambda s: s.shift(1).rolling(21, min_periods=21).mean())
)
vol["realized_vol_trend_21d"] = (
    vol["realized_vol_63d"] - vol["realized_vol_21d_prior_mean"]
)

# 2) Features available at quotation time.
FREQUENCY_TO_MONTHS = {
    "1d": 1 / 30.44,
    "1m": 1, "m": 1, "monthly": 1, "mensual": 1, "1 month": 1,
    "2m": 2,
    "3m": 3, "q": 3, "quarterly": 3, "trimestral": 3, "3 months": 3,
    "6m": 6,
    "1y": 12, "y": 12, "12m": 12, "annual": 12, "anual": 12,
}
def frequency_to_months(s):
    key = s.astype("string").str.strip().str.lower()
    result = key.map(FREQUENCY_TO_MONTHS)
    unknown = sorted(key[result.isna()].dropna().unique())
    if unknown:
        raise ValueError(f"Unmapped frequencies: {unknown}")
    return result.astype(float)

base = rfqs.copy()
base["observation_frequency_months"] = frequency_to_months(base["observation_frequency"])
base["basket_size"] = base["underlyings"].str.split("|").str.len()
base["log_notional_credits"] = np.log1p(base["notional_credits"])
base["requested_year"] = base["requested_date"].dt.year
base["requested_month_sin"] = np.sin(2 * np.pi * base["requested_date"].dt.month / 12)
base["requested_month_cos"] = np.cos(2 * np.pi * base["requested_date"].dt.month / 12)
base["requested_dayofweek"] = base["requested_date"].dt.dayofweek

# 3) One temporary row per RFQ-ticker.
# Reference uses an exact join. Market uses the latest record BEFORE requested_date.
rfq_ticker = (
    base[["rfq_id", "requested_date", "underlyings"]]
    .assign(underlying=lambda x: x["underlyings"].str.split("|"))
    .explode("underlying", ignore_index=True)
    .drop(columns="underlyings")
)
rfq_ticker["underlying"] = rfq_ticker["underlying"].str.strip()
rfq_ticker = rfq_ticker.merge(
    reference, on="underlying", how="left", validate="many_to_one", indicator="reference_match"
)
rfq_ticker = pd.merge_asof(
    rfq_ticker.sort_values(["requested_date", "underlying"]),
    vol.sort_values(["date", "underlying"]),
    left_on="requested_date", right_on="date", by="underlying",
    direction="backward", allow_exact_matches=False,
)
rfq_ticker["market_lag_days"] = (rfq_ticker["requested_date"] - rfq_ticker["date"]).dt.days
assert rfq_ticker["reference_match"].eq("both").all(), "Ticker missing from reference"
assert rfq_ticker["realized_vol_63d"].notna().all(), "RFQ-ticker missing prior market history"
assert rfq_ticker["realized_vol_trend_21d"].notna().all(), "RFQ-ticker missing prior trend history"

# 4) Aggregate back to exactly one row per RFQ.
basket = rfq_ticker.groupby("rfq_id", as_index=False).agg(
    n_underlyings=("underlying", "size"),
    n_market_matches=("realized_vol_63d", "count"),
    realized_vol_min=("realized_vol_63d", "min"),
    realized_vol_max=("realized_vol_63d", "max"),
    realized_vol_mean=("realized_vol_63d", "mean"),
    realized_vol_std=("realized_vol_63d", "std"),
    realized_vol_trend_21d_mean=("realized_vol_trend_21d", "mean"),
    structural_base_vol_min=("structural_base_vol", "min"),
    structural_base_vol_max=("structural_base_vol", "max"),
    structural_base_vol_mean=("structural_base_vol", "mean"),
    structural_base_vol_std=("structural_base_vol", "std"),
    market_lag_days_max=("market_lag_days", "max"),
)
for col in ["realized_vol_std", "structural_base_vol_std"]:
    basket[col] = basket[col].fillna(0.0)  # Single products have no dispersion.
basket["realized_vol_range"] = basket["realized_vol_max"] - basket["realized_vol_min"]
basket["structural_base_vol_range"] = basket["structural_base_vol_max"] - basket["structural_base_vol_min"]
basket["realized_minus_structural_vol"] = basket["realized_vol_mean"] - basket["structural_base_vol_mean"]
basket["market_match_rate"] = basket["n_market_matches"] / basket["n_underlyings"]
assert basket["rfq_id"].is_unique and basket["market_match_rate"].eq(1).all()

# 5) Categorical dummy variables and ticker-presence dummy variables.
# Do not one-hot 2,043 full basket combinations: there are only 14 atomic tickers.
ticker_dummies = (
    pd.crosstab(rfq_ticker["rfq_id"], rfq_ticker["underlying"])
    .clip(upper=1).add_prefix("has_underlying_").reset_index()
)
features = base.merge(basket, on="rfq_id", how="left", validate="one_to_one")
features = features.merge(ticker_dummies, on="rfq_id", how="left", validate="one_to_one")
categoricals = ["product_type", "basket_type", "counterparty", "trader_id"]
features[categoricals] = features[categoricals].fillna("UNKNOWN")
features = pd.get_dummies(features, columns=categoricals, prefix=categoricals, dtype="int8")
assert len(features) == len(rfqs) and features["rfq_id"].is_unique

# 6) Training set: only executed RFQs have the target.
TARGET = "avg_duration_months"
NON_MODEL_COLUMNS = [
    "rfq_id", TARGET, "executed", "underlyings", "observation_frequency",
    "requested_date", "start_date", "end_date",
]
all_rfqs_features = features.copy()
train_features = features.loc[features["executed"]].copy()
# Retained in the processed table for future experiments, but not in the current model contract.
EXPERIMENTAL_ONLY_COLUMNS = ["realized_vol_trend_21d_mean"]
model_feature_columns = [
    c for c in train_features
    if c not in NON_MODEL_COLUMNS and c not in EXPERIMENTAL_ONLY_COLUMNS
]
X = train_features[model_feature_columns]
y = train_features[TARGET]
assert y.notna().all()
assert X.select_dtypes(include="object").empty
assert not X.isna().any().any(), "Model features contain missing values"

# 7) Reproducible outputs for training and inference.
all_rfqs_features.to_csv(PROCESSED_DIR / "all_rfqs_features.csv", index=False)
train_features.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
pd.Series(model_feature_columns, name="feature_name").to_csv(
    PROCESSED_DIR / "model_feature_columns.csv", index=False
)

print(f"Total RFQs: {len(all_rfqs_features):,}")
print(f"Training RFQs: {len(train_features):,}")
print(f"Model features: {X.shape[1]:,}")
print(f"Outputs: {PROCESSED_DIR}")


Total RFQs: 25,000
Training RFQs: 13,796
Model features: 96
Outputs: /home/kiril/Proyectos/starwars_autocalls/data/processed


### Outputs

- `data/processed/all_rfqs_features.csv`: integrated table for all RFQs.
- `data/processed/train_features.csv`: executed RFQs, including the target.
- `data/processed/model_feature_columns.csv`: feature contract for training and inference.
